In [13]:
#import
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(30)
device="cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [14]:
class PositionalEncoding(nn.Module):
  def __init__(self,d_model,max_len=500):
    super().__init__()
    pe=torch.zeros(max_len,d_model)
    position=torch.arange(0,max_len,dtype=torch.float).unsqueeze(1)
    div_term=torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model))
    pe[:,0::2]=torch.sin(position*div_term)
    pe[:,1::2]=torch.cos(position*div_term)
    pe=pe.unsqueeze(0)
    self.register_buffer('pe',pe)

  def forward(self,x):
    seq_len=x.size(1)
    return x+self.pe[:,:seq_len]

In [15]:
#scaled dot product attention
class ScaledDotProductAttention(nn.Module):
  def __init__(self):
    super().__init__()

  def forward(self,Q,K,V,mask=None):
    d_k=Q.size(-1)
    scores=torch.matmul(Q,K.transpose(-2,-1))/math.sqrt(d_k)
    if mask is not None:
      scores=scores.masked_fill(mask==0,-1e9)

    attention=F.softmax(scores,dim=-1)
    output=torch.matmul(attention,V)
    return output,attention


In [16]:
#Multihead Attention
class MultiHeadAttention(nn.Module):
  def __init__(self,d_model,num_heads):
    super().__init__()
    self.num_heads=num_heads
    self.d_model=d_model
    self.d_k=d_model//num_heads
    self.d_v=d_model//num_heads

    self.W_Q=nn.Linear(d_model,d_model)
    self.W_K=nn.Linear(d_model,d_model)
    self.W_V=nn.Linear(d_model,d_model)
    self.W_O=nn.Linear(d_model,d_model)
    self.attention=ScaledDotProductAttention()

  def split_heads(self,x,batch_size):
    x=x.view(batch_size,-1,self.num_heads,self.d_k).transpose(1,2)
    return x

  def combine_heads(self,x,batch_size):
    x=x.transpose(1,2).contiguous()
    return x.view(batch_size,-1,self.d_model)

  def forward(self,Q,K,V,mask=None):
    batch_size=Q.size(0);
    Q=self.W_Q(Q)
    K=self.W_K(K)
    V=self.W_V(V)

    Q=self.split_heads(Q,batch_size)
    K=self.split_heads(K,batch_size)
    V=self.split_heads(V,batch_size)

    output,attention=self.attention(Q,K,V,mask)
    output=self.combine_heads(output,batch_size)
    output=self.W_O(output)
    return output,attention

In [17]:
#Positionwise Feed Forward

class FeedForward(nn.Module):
    def __init__(self,d_model,d_ff):
        super().__init__()

        self.fc1=nn.Linear(d_model,d_ff)
        self.fc2=nn.Linear(d_ff,d_model)
        self.relu=nn.ReLU()

    def forward(self,x):
        return self.fc2(self.relu(self.fc1(x)))

In [18]:
#Encoder Layer

class EncoderLayer(nn.Module):
    def __init__(self,d_model,num_heads,d_ff,dropout=0.1):
        super().__init__()

        self.self_attention=MultiHeadAttention(d_model,num_heads)
        self.feed_forward=FeedForward(d_model,d_ff)

        self.norm1=nn.LayerNorm(d_model)
        self.norm2=nn.LayerNorm(d_model)

        self.dropout=nn.Dropout(dropout)

    def forward(self,x,mask=None):

        attn_output,_=self.self_attention(x,x,x,mask)

        x=self.norm1(x+self.dropout(attn_output))

        ff_output=self.feed_forward(x)

        x=self.norm2(x+self.dropout(ff_output))

        return x

In [19]:
# Decoder Layer

class DecoderLayer(nn.Module):
    def __init__(self,d_model,num_heads,d_ff,dropout=0.1):
        super().__init__()

        self.self_attention=MultiHeadAttention(d_model,num_heads)

        self.cross_attention=MultiHeadAttention(d_model,num_heads)

        self.feed_forward=FeedForward(d_model,d_ff)

        self.norm1=nn.LayerNorm(d_model)
        self.norm2=nn.LayerNorm(d_model)
        self.norm3=nn.LayerNorm(d_model)

        self.dropout=nn.Dropout(dropout)

    def forward(self,x,encoder_output,src_mask=None,tgt_mask=None):

        attn_output,_=self.self_attention(x,x,x,tgt_mask)

        x=self.norm1(x+self.dropout(attn_output))

        cross_output,_=self.cross_attention(
            x,
            encoder_output,
            encoder_output,
            src_mask
        )

        x=self.norm2(x+self.dropout(cross_output))

        ff_output=self.feed_forward(x)

        x=self.norm3(x+self.dropout(ff_output))

        return x

In [20]:
# Encoder

class Encoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model,
        num_layers,
        num_heads,
        d_ff,
        max_len,
        dropout=0.1
    ):
        super().__init__()

        self.embedding=nn.Embedding(vocab_size,d_model)

        self.positional_encoding=PositionalEncoding(d_model,max_len)

        self.layers=nn.ModuleList([
            EncoderLayer(d_model,num_heads,d_ff,dropout)
            for _ in range(num_layers)
        ])

        self.dropout=nn.Dropout(dropout)

        self.d_model=d_model

    def forward(self,x,mask=None):

        x=self.embedding(x)*math.sqrt(self.d_model)

        x=self.positional_encoding(x)

        x=self.dropout(x)

        for layer in self.layers:
            x=layer(x,mask)

        return x

In [21]:
#Decoder

class Decoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model,
        num_layers,
        num_heads,
        d_ff,
        max_len,
        dropout=0.1
    ):
        super().__init__()

        self.embedding=nn.Embedding(vocab_size,d_model)

        self.positional_encoding=PositionalEncoding(d_model,max_len)

        self.layers=nn.ModuleList([
            DecoderLayer(d_model,num_heads,d_ff,dropout)
            for _ in range(num_layers)
        ])

        self.dropout=nn.Dropout(dropout)

        self.d_model=d_model

    def forward(
        self,
        x,
        encoder_output,
        src_mask=None,
        tgt_mask=None
    ):

        x=self.embedding(x)*math.sqrt(self.d_model)

        x=self.positional_encoding(x)

        x=self.dropout(x)

        for layer in self.layers:
            x=layer(
                x,
                encoder_output,
                src_mask,
                tgt_mask
            )

        return x

In [22]:
#Transformer

class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=512,
        num_layers=6,
        num_heads=8,
        d_ff=2048,
        max_len=5000,
        dropout=0.1
    ):
        super().__init__()

        self.encoder=Encoder(
            src_vocab_size,
            d_model,
            num_layers,
            num_heads,
            d_ff,
            max_len,
            dropout
        )

        self.decoder=Decoder(
            tgt_vocab_size,
            d_model,
            num_layers,
            num_heads,
            d_ff,
            max_len,
            dropout
        )

        self.fc_out=nn.Linear(d_model,tgt_vocab_size)

    def generate_src_mask(self,src):

        mask=(src!=0).unsqueeze(1).unsqueeze(2)

        return mask

    def generate_tgt_mask(self,tgt):

        batch_size,tgt_len=tgt.shape

        pad_mask=(tgt!=0).unsqueeze(1).unsqueeze(2)

        causal_mask=torch.tril(
            torch.ones((tgt_len,tgt_len),device=tgt.device)
        ).bool()

        causal_mask=causal_mask.unsqueeze(0).unsqueeze(1)

        return pad_mask & causal_mask

    def forward(self,src,tgt):

        src_mask=self.generate_src_mask(src)

        tgt_mask=self.generate_tgt_mask(tgt)

        encoder_output=self.encoder(src,src_mask)

        decoder_output=self.decoder(
            tgt,
            encoder_output,
            src_mask,
            tgt_mask
        )

        output=self.fc_out(decoder_output)

        return output

In [23]:
src_vocab_size=10000
tgt_vocab_size=10000

model=Transformer(
    src_vocab_size,
    tgt_vocab_size
).to(device)

src=torch.randint(1,1000,(2,10)).to(device)

tgt=torch.randint(1,1000,(2,12)).to(device)

print(src.shape)
print(tgt.shape)

torch.Size([2, 10])
torch.Size([2, 12])


In [24]:
#Forward pass

output=model(src,tgt)

print(output.shape)

# Expected:
# [batch_size,tgt_seq_len,tgt_vocab_size]

torch.Size([2, 12, 10000])


In [25]:
mha=MultiHeadAttention(
    d_model=512,
    num_heads=8
).to(device)

x=torch.randn(2,10,512).to(device)

out,attn=mha(x,x,x)

print("Output shape:",out.shape)
print("Attention shape:",attn.shape)

# Attention shape:
# [batch_size,num_heads,seq_len,seq_len]

Output shape: torch.Size([2, 10, 512])
Attention shape: torch.Size([2, 8, 10, 10])


In [26]:
# parameter count

total_params=sum(
    p.numel() for p in model.parameters()
)

trainable_params=sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Total params:",total_params)
print("Trainable params:",trainable_params)

Total params: 59508496
Trainable params: 59508496
